# Step by step testing

## 1) Core Implementation — Manual Prompt Engineering and Evaluation

### Loading models

In [4]:
pip install -r requirements.txt

   ---------------------------------------- 0.0/5.4 MB ? eta -:--:--
   ---------------------------------------  5.2/5.4 MB 26.9 MB/s eta 0:00:01
   ---------------------------------------- 5.4/5.4 MB 20.8 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.7/3.7 MB 24.2 MB/s  0:00:00
   ---------------------------------------- 0.0/645.5 kB ? eta -:--:--
   ---------------------------------------- 645.5/645.5 kB 15.1 MB/s  0:00:00
   ---------------------------------------- 0.0/625.7 kB ? eta -:--:--
   ---------------------------------------- 625.7/625.7 kB 12.1 MB/s  0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 19.8 MB/s  0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ------------------- -------------------- 6.0/12.3 MB 28.5 MB/s eta 0:00:01
   -------------------------------------- - 11.8

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.2 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
numba 0.62.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
s3fs 2025.10.0 requires fsspec==2025.10.0, but you have fsspec 2026.3.0 which is incompatible.
spyder-kernels 3.1.1 requires ipykernel<7,>=6.29.3, but you have ipykernel 7.2.0 which is incompatible.
streamlit 1.51.0 requires packaging<26,>=20, but you have packaging 26.1 which is incompatible.


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F
import random
import re
from collections import Counter

model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
tokenizer = AutoTokenizer.from_pretrained("gpt2")
MAX_PROMPT_TOKENS = 10
GENERATION_LENGTH = 32
TARGETS = ["grogu", "mando", "kuiil", "peli", "fennec"]

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 17628.13it/s]


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [2]:
print(f"Vocab size: {tokenizer.vocab_size}")  # should be 50257

Vocab size: 50257


See how targets tokenize

In [3]:
for t in TARGETS:
    ids = tokenizer.encode(t)
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"  {t:10s} -> {ids} -> {pieces}")

  grogu      -> [70, 3828, 84] -> ['g', 'rog', 'u']
  mando      -> [22249, 78] -> ['mand', 'o']
  kuiil      -> [74, 9019, 346] -> ['k', 'ui', 'il']
  peli       -> [30242, 72] -> ['pel', 'i']
  fennec     -> [69, 29727, 66] -> ['f', 'enne', 'c']


### Main utility functions

In [4]:
def generate(model, tokenizer, prompt, max_new_tokens=GENERATION_LENGTH):
    """
    Given a text prompt, return GPT-2's continuation as a string.
    
    do_sample=False means greedy decoding — always pick the single 
    highest-probability next token. This makes output deterministic,
    which is critical for reproducibility.
    """
    # Convert prompt string -> tensor of token IDs
    input_ids = tokenizer.encode(prompt, return_tensors="pt")
    attention_mask = torch.ones_like(input_ids) # remove warning message
    
    with torch.no_grad():  # Don't track gradients, we're just generating
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic decoding
            pad_token_id=tokenizer.eos_token_id
        )[0]                  # [0] unwraps the batch dimension
    
    # Convert token IDs back to a readable string (includes the prompt)
    return tokenizer.decode(output_ids)[len(prompt):]

In [5]:
prompt = "The bounty hunter walked into"
output = generate(model, tokenizer, prompt)
print([prompt, output])

['The bounty hunter walked into', ' the office of the FBI\'s New York office and asked for a copy of the FBI\'s new report on the case.\n\n"I was told that the']


**Test Harness**  
Constraints: Prompts must (1) NOT contain the target name, (2) be ≤10 tokens when
tokenized, and (3) cause GPT-2 to generate the target in its output

In [ ]:
def test_case(model, tokenizer, prompt, keyword, max_tokens=MAX_PROMPT_TOKENS):
    """
    Test whether a prompt satisfies all constraints AND generates the target.
    The prompt can be a raw string or a list of token IDs.
    
    Conditions checked:
      1. keyword NOT in prompt (case-insensitive) — no cheating
      2. prompt length <= max_tokens when re-tokenized from string form
      3. keyword appears somewhere in GPT-2's generated output
    
    Returns:
      (passed: bool, message: str)
    """
    # handle token ids and raw strings
    if type(prompt) == list: 
        prompt = tokenizer.decode(prompt)

    # Condition 1: keyword not in prompt
    if keyword.lower() in prompt.lower():
        return (False, 1, f"FAIL [constraint 1]: prompt contains '{keyword}'")
    
    # Condition 2: token length — ALWAYS check re-tokenized length
    actual_length = len(tokenizer.encode(prompt))
    if actual_length > max_tokens:
        return (False, 2, f"FAIL [constraint 2]: {actual_length} tokens > {max_tokens}")
    
    # Condition 3: keyword in generated output
    output = generate(model, tokenizer, prompt)
    # outputted word should be standalone. e.x. "a grogu b" is good, "agrogub" is not
    if re.search(fr"\b{keyword.lower()}\b", output.lower()):
        return (True, 0, f"PASS: output = {repr(output)}")
    else:
        return (False, 3, f"FAIL [constraint 3]: output = {repr(output)}")

### Manual Prompt Engineering and Evaluation

In [11]:
# Part A: Verify baselines fail as expected
print("=== Baseline prompts (should all fail) ===")
passed = False
baselines = ["Star Wars", "baby yoda", "bounty hunter", "din djarin"]
for prompt in baselines:
    for target in TARGETS:
        passed, _, _ = test_case(model, tokenizer, prompt, target)
        if passed:  # Flag any surprising passes
            print(f"  UNEXPECTED PASS: '{prompt}' -> {target}")
            passed = True

if not passed:
    print("All cases failed")

=== Baseline prompts (should all fail) ===
All cases failed


#### Substring/Prefix Inspired Prompts

The hypothesis here is that we can get GPT-2 to follow a pattern and produce the target word by using substrings of target characters as the prompt

In [63]:
prompt = "4:1 Water to Rum"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[0]) # target = grogu

['4:1 Water to Rum', ' (1)\n\n1:1 Water to Rum (1)\n\n1:1 Water to Rum (1)\n\n1:1 Water to']


(False,
 "FAIL [constraint 3]: output = ' (1)\\n\\n1:1 Water to Rum (1)\\n\\n1:1 Water to Rum (1)\\n\\n1:1 Water to'")

In [46]:
prompt = "odo guu mand"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[1]) # target = mando

['odo guu mand', 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,']


(True,
 "PASS: output = 'o, yu mando, yu mando, yu mando, yu mando, yu mando, yu mando,'")

In [47]:
prompt = "iil qiil kui"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[2]) # target = kuiil

['iil qiil kui', 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q']


(True,
 "PASS: output = 'il qiil kuiil qiil kuiil qiil kuiil qiil kuiil qiil kuiil q'")

In [48]:
prompt = "i u li gr eli pel"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[3]) # target = peli

['i u li gr eli pel', 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr']


(True,
 "PASS: output = 'i u li gr eli peli u li gr eli peli u li gr eli peli u li gr eli peli u li gr'")

In [49]:
prompt = "nnec fnec-fx ennec fen"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[4]) # target = fennec

['nnec fnec-fx ennec fen', 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f']


(True,
 "PASS: output = 'nec fennec fennec fennec fennec fennec fennec fennec fennec fennec fennec f'")

Initially, the raw substrings alone were not enough to produce the target results. For example, the prompt 'gr ogu' produced the output of 'o, oguo oguo, oguo oguo, oguo oguo, oguo oguo, ogu'. Now this output is somewhat nonsensical, however it is clear to see that there is a pattern of the last characters of the prompt being repeated multiple times. Through manual fiddling, it was found that GPT-2 paid a lot of attention to the tokens on the ends of the prompt and would often repeat those characters in a nonsensical prompt. Thus is made it possible to use substrings of characters of the target word on the ends of the prompt in order to force GPT-2 to recreate the target word. There was interesting behavior for the importance of tokens between the ends of the prompt as well. For the most part, the middle tokens served to force GPT-2 to interpret the prompt as nonsensical and thus be prone to repeating characters of the prompt (sometimes without middle tokens, the output would be real human language). The most significant role the middle tokens played however, was to tune how and where characters appeared in the output. Using these principles and a lot of trial and error, prompts were manually fine tuned to reach all target words showing that this strategy is valid. In terms of success rate, each keyword individually took (approximately) less than 20 prompts in order to find success. 

#### Semantic-Based Prompts

These prompts attempt to use existing semantic associations to generate the desired sequence of characters

In [159]:
prompt = "The Sardinian word for yellow is "
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[0]) # target = grogu

['The Sardinian word for yellow is ', 'ānānā, which means "to be yellow."\n\nThe word for yellow is also used in the same way in the English language. The']


(False,
 'FAIL [constraint 3]: output = \'ānānā, which means "to be yellow."\\n\\nThe word for yellow is also used in the same way in the English language. The\'')

In [150]:
prompt = "Famous manddolinist"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[1]) # target = mando

['Famous manddolinist', ', and the first to play the piano, and the first to play the piano solo.\n\nThe first time I heard the mandolin, I was in']


(True,
 "PASS: output = ', and the first to play the piano, and the first to play the piano solo.\\n\\nThe first time I heard the mandolin, I was in'")

In [157]:
prompt = "is kui sick? yes,"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[2]) # target = kuiil

['is kui sick? yes,', " I'm sick.\n\nKui: I'm sick.\n\nKui: I'm sick.\n\nKui: I'm sick.\n"]


(False,
 'FAIL [constraint 3]: output = " I\'m sick.\\n\\nKui: I\'m sick.\\n\\nKui: I\'m sick.\\n\\nKui: I\'m sick.\\n"')

In [116]:
prompt = "My favorite band is Led"
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[3]) # target = peli

['My favorite band is Led', ' Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin']


(True,
 "PASS: output = ' Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin. I love Led Zeppelin'")

In [158]:
prompt = "The fenec fox "
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[4]) # target = fennec

['The fenec fox ', '\xa0is a very common species in the wild. It is a very common species in the wild. It is a very common species in the wild.\nThe']


(False,
 "FAIL [constraint 3]: output = '\\xa0is a very common species in the wild. It is a very common species in the wild. It is a very common species in the wild.\\nThe'")

This prompting technique can work in certain sircumstances, but is highly dependent on the desired keyword, the strength of the model, and the alotted number of tokens. With this prompting technique, 2/5 of the targets were able to be found. For the target 'peli,' I noticed that "zepelin" contains the target as a substring, and that "Led Zepelin" is a very famous band, so by simply getting the prompt to finish the band name, I was able to get the target. Likewise, I was able to get the target 'mando' by finding a prompt which generated "mandolin," which has the target as a substring. This was a bit trickier, but I found that by misspelling "mandolinist" in such a way that it no longer contained the substring "mando," the model still properly interpreted it as "mandolin musician." This technique failed for 'grogu,' 'kuiil,' and 'fennec.' This is because these substrings are very uncommon as substrings of English words, and even if one was found, the model likely would not know that word. For 'grogu,' I found via Wiktionary that "grogu" is the Sardinian word for "yellow," but it does not appear that the model has a sufficient Sardinian corpus for this technique to work. This problem would be true for the other meanings in other languages. This techinique would likely be more effective on more advanced models with more data. For 'kuiil,' this has the same problem, with the added detriment of "kuiil" not being a word in any language. To this end, I attempted to craft a situation in which the person Kui has a sickness, attempting to leverage the fact that "kuiil" could be formed by combining the words "Kui" and "ill." However, even if I could force these tokens to appear, the amount of tokens alotted means that it would be exceedingly difficult to get the words to not only appear, but to appear inmmediately next to each other without a space inbetween. For the target 'fennec,' since "fennec" is not a substring of any English words, I attempted to force the model to generate something related to Fennec Foxes, an African fox species with distinctive large ears. However, the main hinderance for this strategy was that the model did not appear to know about this species of fox. Despite having several methods of precisely identifying Fennec Foxes in particular -- including the scientific name, characters based on the animal, features of the animal, and even using the same misspelling tactic as worked for "mandolin" -- the model would never generate "fennec" in relation to foxes. This would likely be solved if we used a model with more data, as it would likely have data relating to Fennec Foxes in the corpus.

### ======== TODO ===========
Add 4 more manual prompting strategies similar to that of above. You can add a new header/section by adding a markdown cell and adding `#### Section Name` and then putting your code there.  (P.S. a newline in markdown is two spaces at the end of the line)  
Note that, "For EACH approach: document the exact prompt, show GPT-2’s output, explain your hypothesis, and analyze why it worked/failed. ... Report success rate, insights from manual exploration, and examples of discovered prompts"

## 3) Automated Prompt Search 

In [99]:
def compute_loss(model, tokenizer, prompt_ids, target):
    """
    Compute -log P(target tokens | prompt_ids).
    prompt_ids: 1D tensor of token IDs for the prompt.
    target: string like "grogu"
    """
    target_ids = tokenizer.encode(target)  # e.g. [70, 3828, 84] for "grogu"
    
    # Build full input: prompt + target tokens
    full_ids = torch.tensor([prompt_ids + target_ids])  # shape: [1, seq_len]
    
    with torch.no_grad():
        outputs = model(full_ids)
        logits = outputs.logits  # shape: [1, seq_len, 50257]
    
    # We only care about predicting the target tokens.
    # The model predicts token[i+1] from position i, so:
    # - position len(prompt)-1 predicts target_ids[0]
    # - position len(prompt)   predicts target_ids[1]  (if multi-token target)
    
    loss = 0.0
    prompt_len = len(prompt_ids)
    for i, tid in enumerate(target_ids):
        # logits at position (prompt_len - 1 + i) predicts target_ids[i]
        logit_at_pos = logits[0, prompt_len - 1 + i, :]  # shape: [50257]
        loss += F.cross_entropy(logit_at_pos.unsqueeze(0), torch.tensor([tid]))
    
    return loss.item()

In [100]:
def compute_token_gradients(model, tokenizer, prompt_ids, target):
    """
    Returns: grad tensor of shape [prompt_len, 768]
    Each row is dL/d(embedding of token at that position).
    """
    target_ids = tokenizer.encode(target)
    
    # Get the embedding matrix — shape [50257, 768]
    embed_matrix = model.transformer.wte.weight  # GPT-2 specific
    
    # Look up embeddings for prompt tokens and enable grad tracking
    prompt_tensor = torch.tensor([prompt_ids])
    prompt_embeds = embed_matrix[prompt_tensor].detach().requires_grad_(True)
    # shape: [1, prompt_len, 768]
    
    # Also embed target tokens (no grad needed here)
    target_tensor = torch.tensor([target_ids])
    target_embeds = embed_matrix[target_tensor].detach()
    
    # Concatenate prompt + target embeddings
    full_embeds = torch.cat([prompt_embeds, target_embeds], dim=1)
    
    # Forward pass using embeddings directly (bypass token ID lookup)
    outputs = model(inputs_embeds=full_embeds)
    logits = outputs.logits  # [1, seq_len, 50257]
    
    # Compute loss on target positions
    loss = 0.0
    prompt_len = len(prompt_ids)
    for i, tid in enumerate(target_ids):
        logit_at_pos = logits[0, prompt_len - 1 + i, :]
        loss += F.cross_entropy(logit_at_pos.unsqueeze(0), torch.tensor([tid]))
    
    # Backprop to get dL/d(prompt_embeds)
    loss.backward()
    
    # grad shape: [1, prompt_len, 768] → squeeze to [prompt_len, 768]
    return prompt_embeds.grad.squeeze(0)

In [101]:
def get_top_k_candidates(grad, embed_matrix, k=20):
    """
    For a single token position:
    grad: shape [768] — gradient at that position
    embed_matrix: shape [50257, 768] — all vocab embeddings
    Returns: top-k token IDs most likely to reduce loss
    """
    # Score every vocab token by how well it aligns with -gradient
    # Higher score = better candidate = more likely to reduce loss
    scores = -grad @ embed_matrix.T  # shape: [50257]
    
    # Return indices of top-k highest scores
    top_k_ids = scores.topk(k).indices.tolist()
    return top_k_ids

In [128]:
def find_best_replacement(model, tokenizer, prompt_ids, target, position, k=20):
    """
    At a given position in the prompt, find the token replacement 
    that minimally reduces loss.
    Returns: (best_token_id, best_loss)
    """
    embed_matrix = model.transformer.wte.weight.detach()
    
    # Step 1: get gradient at this position
    grads = compute_token_gradients(model, tokenizer, prompt_ids, target)
    grad_at_pos = grads[position]  # shape: [768]
    
    # Step 2: get top-k candidates
    candidates = get_top_k_candidates(grad_at_pos, embed_matrix, k=k)
    
    best_loss = float('inf')
    best_token = prompt_ids[position]  # default: keep current token
    constraint_tracker = Counter({1:0, 2:0, 3:0, 'total_eval': 0})
    
    for candidate_id in candidates:
        constraint_tracker['total_eval'] += 1
        # Build new prompt with this candidate substituted in
        new_prompt = prompt_ids[:position] + [candidate_id] + prompt_ids[position+1:]
        
        # --- Constraint check (tokenization stability) ---
        decoded = tokenizer.decode(new_prompt)
        if target.lower() in decoded.lower():
            constraint_tracker[1] += 1
            continue  # Can't have target in prompt
        if len(tokenizer.encode(decoded)) > 10:
            constraint_tracker[2] += 1
            continue  # Token budget exceeded after round-trip

        # Step 3: compute actual loss for this candidate
        loss = compute_loss(model, tokenizer, new_prompt, target)
        
        if loss < best_loss:
            best_loss = loss
            best_token = candidate_id
    
    # no valid candidate was found
    if best_token == prompt_ids[position] and best_loss == float('inf'):
        constraint_tracker[3] += 1
        return None, float('inf'), constraint_tracker  # signal: no valid candidate at this position

    return best_token, best_loss, constraint_tracker

In [129]:
def gradient_guided_search(model, tokenizer, target, max_iters=200, k=20, seed=42, patience=5):
    """
    Main gradient-guided discrete search loop.
    patience: how many full iterations without local improvement
    before triggering a restart. Give each starting point
    a fair chance to explore before giving up on it.
    Returns: best prompt string found, or None if unsuccessful.
    """
    torch.manual_seed(seed)
    random.seed(seed)
    
    # Initialize with random valid prompt (no target keyword, <=10 tokens)
    prompt_ids = initialize_random_prompt(tokenizer, target, max_tokens=10)
    constraint_tracker = Counter({1:0, 2:0, 3:0, 'total_eval': 0})
    prompt_count = 0

    # --- Two separate loss trackers ---
    global_best_loss = float('inf')   # best seen across ALL restarts — for reporting
    global_best_prompt = None
    
    local_best_loss = float('inf')    # best seen since last restart — for restart logic
    iters_without_local_improvement = 0
    
    for iteration in range(max_iters):
        improved_locally = False
        
        for pos in range(len(prompt_ids)):
            new_token, new_loss, constraints = find_best_replacement(
                model, tokenizer, prompt_ids, target, pos, k=k
            )
            constraint_tracker += constraints
            
            if (new_token is not None) and (new_loss < local_best_loss):
                prompt_ids[pos] = new_token
                local_best_loss = new_loss
                improved_locally = True
                
                # Also update global best if this is an all-time best
                if new_loss < global_best_loss:
                    global_best_loss = new_loss
                    global_best_prompt = tokenizer.decode(prompt_ids)
        
        # Check for success
        decoded = tokenizer.decode(prompt_ids)
        passed, constraint, _ = test_case(model, tokenizer, decoded, target)
        prompt_count += 1
        
        if constraint > 0:
            constraint_tracker[constraint] += 1
        if passed:
            print(f"  Found at iteration {iteration}: '{decoded}'")
            return decoded, constraint_tracker, prompt_count
        
        if iteration % 10 == 0:
            print(f"  Iter {iteration}: local={local_best_loss:.4f}, "
                  f"global={global_best_loss:.4f}, prompt='{decoded}'")
        
        # --- Patience-based restart logic ---
        if not improved_locally:
            iters_without_local_improvement += 1
        else:
            iters_without_local_improvement = 0  # reset patience counter on progress
        
        if iters_without_local_improvement >= patience:
            prompt_ids = initialize_random_prompt(tokenizer, target, max_tokens=10)
            local_best_loss = float('inf')        # fresh slate for new starting point
            iters_without_local_improvement = 0
            print(f"  Iter {iteration}: patience exceeded, restarting "
                  f"(global best so far: {global_best_loss:.4f})")
    
    # Return global best even if it never passed test_case
    # (useful for near-miss analysis in your error analysis section)
    return global_best_prompt, constraint_tracker, prompt_count


def initialize_random_prompt(tokenizer, target, max_tokens=10):
    """Generate a random valid starting prompt."""
    while True:
        # Sample random tokens from the vocabulary
        ids = [random.randint(0, tokenizer.vocab_size - 1) for _ in range(max_tokens)]
        decoded = tokenizer.decode(ids)
        # Check constraints
        if target.lower() not in decoded.lower():
            if len(tokenizer.encode(decoded)) <= max_tokens:
                return ids

In [84]:
x = gradient_guided_search(model, tokenizer, TARGETS[0]) # grogu

  Iter 0: local=12.9773, global=12.9773, prompt='reen demonicpauseagar Gul Shu Freeatibleapp guilty'
  Found at iteration 5: 'errilla demonicgrave GIg Rogu Recommendedion courtroom'


In [109]:
x = gradient_guided_search(model, tokenizer, TARGETS[1]) # mando

  Iter 0: local=7.5667, global=7.5667, prompt=' Gru NicaraguaYouInsp MOR tacosAM moodhern Hel'
  Found at iteration 1: ' Jos Nicaragua Legislative Yuan WRITEkosldeomo har Hel'


In [70]:
x = gradient_guided_search(model, tokenizer, TARGETS[2]) # kuiil

  Iter 0: loss=14.5157, prompt=' phosphateCLASSIFIEDdial� Ik Finnishcultural Ike auī'
  Found at iteration 2: 'jarukeduiil Kau00200000taboolaKar auī'


In [72]:
x = gradient_guided_search(model, tokenizer, TARGETS[3]) # peli

  Iter 0: loss=9.2223, prompt=' Siem DelePicture MilanPLBel Hipraft tcp Sim'
  Found at iteration 3: ' lawyersč slides Macedonia coffinBrow Cancel200000 ka Sim'


In [74]:
x = gradient_guided_search(model, tokenizer, TARGETS[4]) # fennec

  Iter 0: loss=10.9324, prompt='perture FitzpatrickYouéeNVIDIA Elise globalelo tcp greedy'
  Found at iteration 2: 'azy Fen Fou FormatnoticeTesla globalgirlZip greedy'


In [75]:
prompt = x
output = generate(model, tokenizer, prompt)
print([prompt, output])
test_case(model, tokenizer, prompt, TARGETS[4]) # target = grogu

['azy Fen Fou FormatnoticeTesla globalgirlZip greedy', 'fennec Tidalfennec Tidalfennec - Fennec Tidalfennec - Fennec Tidalfennec -']


(True,
 "PASS: output = 'fennec Tidalfennec Tidalfennec - Fennec Tidalfennec - Fennec Tidalfennec -'")

## 4) Error Analysis

In [130]:
def evaluate_target(model, tokenizer, target, n_searches=20, n_repro=10):
    """
    Full evaluation for one target keyword.
    
    n_searches: how many independent search runs to attempt
                (answers: "how reliably does the search work?")
    n_repro:    how many times to re-run each successful prompt
                (answers: "how stable is the prompt once found?")
    
    Returns a results dict ready for your report table.
    """
    search_results = []
    constraint_tracker = Counter({1:0, 2:0, 3:0, 'total_eval': 0})
    
    for i in range(n_searches):
        seed = i * 11  # different seed = different random starting point
        prompt, constraints, prompt_count = gradient_guided_search(
            model, tokenizer, target,
            max_iters=200, k=20, seed=seed
        )

        constraint_tracker += constraints
        
        if prompt is not None:
            # --- Measurement 2: reproducibility ---
            repro_passes = sum(
                test_case(model, tokenizer, prompt, target)[0]
                for _ in range(n_repro)
            )
            repro_rate = repro_passes / n_repro
            
            search_results.append({
                "found": True,
                "prompt": prompt,
                "prompt_count": prompt_count,
                "repro_rate": repro_rate,
            })
        else:
            search_results.append({"found": False})
    
    # --- Measurement 1: search success rate ---
    n_found = sum(r["found"] for r in search_results)
    search_success_rate = n_found / n_searches
    
    # --- Classify outcome ---
    if n_found == 0:
        category = "Complete Failure"
    else:
        successful = [r for r in search_results if r["found"]]
        avg_repro = sum(r["repro_rate"] for r in successful) / len(successful)
        best_prompt = max(successful, key=lambda r: r["repro_rate"])
        
        # Check for near miss: does GPT-2 generate something close but wrong?
        near_miss = check_near_miss(model, tokenizer, target)
        
        if avg_repro >= 0.8:
            category = "Complete Success"
        elif near_miss:
            category = "Near Miss"
        else:
            category = "Partial Success"
    
    return {
        "target": target,
        "search_success_rate": search_success_rate,
        "n_found": n_found,
        "n_searches": n_searches,
        "avg_repro_rate": avg_repro if n_found > 0 else 0.0,
        "best_prompt": best_prompt["prompt"] if n_found > 0 else None,
        "total_prompt_count": sum(r['prompt_count'] for r in search_results),
        "category": category,
        "total_constr1": constraint_tracker[1],
        "total_constr2": constraint_tracker[2],
        "total_constr3": constraint_tracker[3],
        "total_eval": constraint_tracker['total_eval'],
    }


def check_near_miss(model, tokenizer, target, n_attempts=10):
    """
    Check if GPT-2 produces outputs that are close to the target
    (e.g. "grog" instead of "grogu") even when the full target isn't found.
    Uses simple prefix matching as a heuristic.
    """
    # Try a few random short prompts and see if output starts with target prefix
    prefix = target[:3]  # e.g. "gro" for "grogu"
    for _ in range(n_attempts):
        ids = [random.randint(0, tokenizer.vocab_size - 1) for _ in range(5)]
        prompt = tokenizer.decode(ids)
        output = generate(model, tokenizer, prompt)
        if prefix.lower() in output.lower() and target.lower() not in output.lower():
            return True
    return False

In [ ]:
evaluate_target(model, tokenizer, TARGETS[0])

In [ ]:
# 10m 27.9s
# {'target': 'mando',
#  'search_success_rate': 1.0,
#  'n_found': 20,
#  'n_searches': 20,
#  'avg_repro_rate': 1.0,
#  'best_prompt': 'mandiflower Spo asphaltRepeMand indo Nicaragua comprom pronounce',
#  'total_prompt_count': 36,
#  'category': 'Complete Success',
#  'total_constr1': 81,
#  'total_constr2': 452,
#  'total_constr3': 16,
#  'total_eval': 7200}
evaluate_target(model, tokenizer, TARGETS[1])

In [ ]:
# 21m 13.4s 
# {'target': 'peli',
#  'search_success_rate': 1.0,
#  'n_found': 20,
#  'n_searches': 20,
#  'avg_repro_rate': 1.0,
#  'best_prompt': 'hammad mipelhei?????-?????- KN pelPal pronunciation Ves',
#  'total_prompt_count': 87,
#  'category': 'Complete Success',
#  'total_constr1': 492,
#  'total_constr2': 1048,
#  'total_constr3': 67,
#  'total_eval': 17400}
evaluate_target(model, tokenizer, TARGETS[3])

In [ ]:
# 19m 27.4s
# {'target': 'fennec',
#  'search_success_rate': 1.0,
#  'n_found': 20,
#  'n_searches': 20,
#  'avg_repro_rate': 1.0,
#  'best_prompt': ' Blasio TradablefeTraEStream Quadrojohnser Tradable Pound',
#  'total_prompt_count': 58,
#  'category': 'Complete Success',
#  'total_constr1': 0,
#  'total_constr2': 1073,
#  'total_constr3': 38,
#  'total_eval': 11600}

evaluate_target(model, tokenizer, TARGETS[4])

  Iter 0: local=7.6532, global=7.6532, prompt=' Pixie Tradablefe Quadro gifwalletjohnarchives Tradable Pound'
  Found at iteration 4: ' Blasio TradablefeTraEStream Quadrojohnser Tradable Pound'
  Iter 0: local=8.3296, global=8.3296, prompt='avorite Fem crit Fen Elsa Hexolinaasionally Petersonneutral'
  Found at iteration 1: 'BuyableInstoreAndOnline Fossategor Fen ElsaFre cubasionallyuberneutral'
